# PaperStreet Research

Interactive research notebook for exploring market data and strategy logic.

**Before running:** ensure TWS is open and connected on port 7497.

**Workflow:** run the Setup cell once per session. All other cells can be re-run
independently without reconnecting.

## 1. Setup
Connect to TWS once. Re-running this cell will reconnect if the session was closed.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

from research.session import Session

# Disconnect any existing session before creating a new one
try:
    session.disconnect()
except NameError:
    pass

session = Session()
print("Session ready. TWS connected:", session.is_connected)

## 2. Fetch Data
Pull daily bars for a symbol. Adjust `SYMBOL` and `DURATION` to explore different instruments.

In [ ]:
SYMBOL   = "SPY"
DURATION = "3 M"   # 1 W | 1 M | 3 M | 6 M | 1 Y | 2 Y | 5 Y

df = session.market_data.get_daily_bars(SYMBOL, duration=DURATION)

print(f"\n{SYMBOL} — {len(df)} bars ({DURATION})")
print(f"From : {df.index[0].date()}")
print(f"To   : {df.index[-1].date()}")
print(f"\nShape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
print(f"Dtypes  :\n{df.dtypes}")
df.head()

## 3. Price & Volume Chart
OHLC bar chart with volume subplot.

In [ ]:
fig = plt.figure(figsize=(14, 7))
gs  = gridspec.GridSpec(2, 1, height_ratios=[3, 1], hspace=0.08)

ax_price  = fig.add_subplot(gs[0])
ax_volume = fig.add_subplot(gs[1], sharex=ax_price)

# --- Price: close line + high/low range shaded ---
ax_price.plot(df.index, df["close"], color="#1f77b4", linewidth=1.5, label="Close")
ax_price.fill_between(df.index, df["low"], df["high"], alpha=0.15, color="#1f77b4", label="High/Low range")
ax_price.set_ylabel("Price (USD)")
ax_price.set_title(f"{SYMBOL} — Daily Price & Volume ({DURATION})")
ax_price.legend(loc="upper left")
ax_price.grid(True, alpha=0.3)
plt.setp(ax_price.get_xticklabels(), visible=False)

# --- Volume ---
ax_volume.bar(df.index, df["volume"], color="#aec7e8", width=0.8)
ax_volume.set_ylabel("Volume")
ax_volume.grid(True, alpha=0.3)
ax_volume.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax_volume.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

## 4. Indicator Analysis — SMA + Volatility Bands

Computes the same indicators your `MeanReversionStrategy` uses internally:
- **SMA** (fair value)
- **Volatility bands** (SMA ± spread_multiplier × rolling std dev)

Adjust `WINDOW` and `SPREAD_MULTIPLIER` to see how the entry thresholds shift.

In [ ]:
WINDOW           = 20
SPREAD_MULTIPLIER = 0.5

df["sma"]        = df["close"].rolling(WINDOW).mean()
df["volatility"] = df["close"].rolling(WINDOW).std()
df["upper_band"] = df["sma"] + SPREAD_MULTIPLIER * df["volatility"]
df["lower_band"] = df["sma"] - SPREAD_MULTIPLIER * df["volatility"]
df["deviation"]  = df["close"] - df["sma"]

print(f"Window: {WINDOW} bars | Spread multiplier: {SPREAD_MULTIPLIER}")
print(f"\nCurrent deviation from fair value : {df['deviation'].iloc[-1]:.4f}")
print(f"Current threshold (±)             : {(SPREAD_MULTIPLIER * df['volatility'].iloc[-1]):.4f}")
df[["close", "sma", "upper_band", "lower_band", "deviation"]].tail(10)

## 5. Signal Overlay
Visualize where the mean reversion strategy would have generated BUY and SELL signals on the fetched data.

In [ ]:
# Derive signals from the indicator columns computed above
# Mirrors the on_bar logic in MeanReversionStrategy exactly
df["signal"] = None
position = 0
MAX_POSITION = 50
ORDER_SIZE   = 10

for i in range(len(df)):
    row = df.iloc[i]
    if pd.isna(row["sma"]):
        continue

    deviation = row["deviation"]
    threshold = SPREAD_MULTIPLIER * row["volatility"]

    if deviation < -threshold and position < MAX_POSITION:
        df.at[df.index[i], "signal"] = "BUY"
        position += ORDER_SIZE
    elif deviation > threshold and position > 0:
        df.at[df.index[i], "signal"] = "SELL"
        position = max(0, position - ORDER_SIZE)

buy_signals  = df[df["signal"] == "BUY"]
sell_signals = df[df["signal"] == "SELL"]

print(f"BUY signals : {len(buy_signals)}")
print(f"SELL signals: {len(sell_signals)}")

# --- Plot ---
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df.index, df["close"],      color="#1f77b4",  linewidth=1.5, label="Close",      zorder=2)
ax.plot(df.index, df["sma"],        color="#ff7f0e",  linewidth=1.2, label=f"SMA({WINDOW})", zorder=2)
ax.plot(df.index, df["upper_band"], color="#2ca02c",  linewidth=0.8, linestyle="--", label="Upper band", zorder=2)
ax.plot(df.index, df["lower_band"], color="#d62728",  linewidth=0.8, linestyle="--", label="Lower band", zorder=2)
ax.fill_between(df.index, df["lower_band"], df["upper_band"], alpha=0.06, color="gray")

ax.scatter(buy_signals.index,  buy_signals["close"],  marker="^", color="#2ca02c", s=100, zorder=3, label=f"BUY ({len(buy_signals)})")
ax.scatter(sell_signals.index, sell_signals["close"], marker="v", color="#d62728", s=100, zorder=3, label=f"SELL ({len(sell_signals)})")

ax.set_title(f"{SYMBOL} — Mean Reversion Signals (window={WINDOW}, multiplier={SPREAD_MULTIPLIER})")
ax.set_ylabel("Price (USD)")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Multi-Symbol Comparison
Compare relative performance and correlation across a basket of symbols.
Adjust `SYMBOLS` and `DURATION` freely.

In [ ]:
SYMBOLS  = ["SPY", "QQQ", "IWM"]
DURATION = "3 M"

closes = session.market_data.get_close_prices(SYMBOLS, duration=DURATION)

print(f"Fetched {len(closes)} aligned dates across {SYMBOLS}")
closes.tail()

In [ ]:
# Normalise to 100 at the start so all series are comparable on the same axis
normalised = (closes / closes.iloc[0]) * 100

fig, ax = plt.subplots(figsize=(14, 5))

for symbol in SYMBOLS:
    ax.plot(normalised.index, normalised[symbol], linewidth=1.5, label=symbol)

ax.axhline(100, color="gray", linewidth=0.8, linestyle="--")
ax.set_title(f"Relative Performance — Normalised to 100 ({DURATION})")
ax.set_ylabel("Indexed Price")
ax.legend()
ax.grid(True, alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
ax.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix of daily returns
returns = closes.pct_change().dropna()
corr    = returns.corr()

print("Daily return correlation matrix:")
print(corr.round(4))

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(corr, vmin=-1, vmax=1, cmap="RdYlGn")
plt.colorbar(im, ax=ax)

ax.set_xticks(range(len(SYMBOLS)))
ax.set_yticks(range(len(SYMBOLS)))
ax.set_xticklabels(SYMBOLS)
ax.set_yticklabels(SYMBOLS)

for i in range(len(SYMBOLS)):
    for j in range(len(SYMBOLS)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=10)

ax.set_title("Return Correlation")
plt.tight_layout()
plt.show()

## 7. Teardown
Disconnect from TWS when you are done. Safe to skip if you plan to keep the session open.

In [ ]:
session.disconnect()
print("Disconnected. Session active:", session.is_connected)